In [2]:
from model import get_dataloader, C4Dataset
import os
import torch

In [3]:
os.environ["WORLD_SIZE"] = "1"
os.environ["RANK"] = "0"

In [4]:

dataloader_1 = get_dataloader(
    dataset_type="c4",
    dataset_path="data",
    dataset_split="train",
    total_batch_size=10,
    sequence_length=32,
    num_workers=0,
    seed=1,
    world_size_independent=False,
    use_new_sampling_method=True,
    shuffle=True,
)

next(iter(dataloader_1))

tensor([[  312, 37217,   284,  7301,   262,  1440, 14371,   286,   262,  5440,
            13,   887,   287,   281,  2479,   810,  3067,   318,   517,  9857,
           621,  1683,    11,   644,   857,   340,  1107,  1612,   284,   423,
           587,   564,   246],
        [ 6593,  5952,   287, 14493,    11,   968,  1971,    11,   290, 10140,
           389,  1022,   642,    25,    16,   290,   767,    25,    16,    13,
          2893, 41432, 26297,  6593,   743,  6133,  2585,   287,  8868, 13594,
         41528,   592,   290],
        [ 1002,   523,    11,   345,   389,   287,   262,   826,  1295,    11,
           356,  2630,   428,  1281,   329,   345,     0,   554,  2142,   352,
            11,   356,   447,   247,   260,  1016,   284,  2962,   319,   617,
           286,   262,  1243],
        [ 8472,  8557,  2891,    13, 25134,  9084,  1077,   291, 24075,    11,
          8798,   262, 10074,   286,  3406,  8653,    11,   318,   763,    12,
         39351,   351,   607,  6621,  

In [5]:
ole = iter(dataloader_1)

In [6]:
next(iter(dataloader_1))

tensor([[  312, 37217,   284,  7301,   262,  1440, 14371,   286,   262,  5440,
            13,   887,   287,   281,  2479,   810,  3067,   318,   517,  9857,
           621,  1683,    11,   644,   857,   340,  1107,  1612,   284,   423,
           587,   564,   246],
        [ 6593,  5952,   287, 14493,    11,   968,  1971,    11,   290, 10140,
           389,  1022,   642,    25,    16,   290,   767,    25,    16,    13,
          2893, 41432, 26297,  6593,   743,  6133,  2585,   287,  8868, 13594,
         41528,   592,   290],
        [ 1002,   523,    11,   345,   389,   287,   262,   826,  1295,    11,
           356,  2630,   428,  1281,   329,   345,     0,   554,  2142,   352,
            11,   356,   447,   247,   260,  1016,   284,  2962,   319,   617,
           286,   262,  1243],
        [ 8472,  8557,  2891,    13, 25134,  9084,  1077,   291, 24075,    11,
          8798,   262, 10074,   286,  3406,  8653,    11,   318,   763,    12,
         39351,   351,   607,  6621,  

In [7]:

dataloader_1 = get_dataloader(
    dataset_type="c4",
    dataset_path="data",
    dataset_split="train",
    total_batch_size=10,
    sequence_length=32,
    num_workers=0,
    seed=1,
    world_size_independent=True,
    use_new_sampling_method=True,
    shuffle=True,
)


dataloader_2 = get_dataloader(
    dataset_type="c4",
    dataset_path="data",
    dataset_split="train",
    total_batch_size=10,
    sequence_length=32,
    num_workers=0,
    seed=2,
    world_size_independent=False,
    use_new_sampling_method=True,
    shuffle=True,
)

In [8]:
next(iter(dataloader_2)).shape

torch.Size([10, 33])

In [9]:
import torch

class CustomCombinedLoader:
    def __init__(self, loader1, loader2, final_seq_len=None, final_batch_size=None):
        self.loader1 = loader1
        self.loader2 = loader2
        self.final_seq_len = final_seq_len
        self.final_batch_size = final_batch_size

    def __iter__(self):
        self.iter1 = iter(self.loader1)
        self.iter2 = iter(self.loader2)
        return self

    def __next__(self):
        batch1 = next(self.iter1)
        batch2 = next(self.iter2)

        return self._combine_batches(batch1, batch2)

    def _combine_batches(self, batch1, batch2):
        def truncate(tensor):
            # Apply batch size truncation
            if self.final_batch_size is not None:
                tensor = tensor[:self.final_batch_size]
            # Apply sequence length truncation (assuming sequences are at dim 1)
            if self.final_seq_len is not None and tensor.ndim > 1:
                tensor = tensor[:, :self.final_seq_len]
            return tensor

        if isinstance(batch1, tuple):
            combined = tuple(torch.cat([b1, b2]) for b1, b2 in zip(batch1, batch2))
            return tuple(truncate(t) for t in combined)
        elif isinstance(batch1, dict):
            combined = {k: torch.cat([batch1[k], batch2[k]]) for k in batch1}
            return {k: truncate(v) for k, v in combined.items()}
        else:
            combined = torch.cat([batch1, batch2])
            return truncate(combined)

    # def __len__(self):
    #     return min(len(self.loader1), len(self.loader2))


In [ ]:
# class TruncateOnTheFlyLoader:
#     def __init__(self, dataloader,  final_seq_len=None, final_batch_size=None):
#         self.loader
#         self.final_seq_len = final_seq_len
#         self.final_batch_size = final_batch_size

#     def __iter__(self):
#         self.iter1 = iter(self.loader1)
#         self.iter2 = iter(self.loader2)
#         self.iter = iter(self.loader)
#         return self

#     def __next__(self):
#         batch1 = next(self.iter1)
#         batch2 = next(self.iter2)

#         return self._combine_batches(batch1, batch2)

#     def _combine_batches(self, batch1, batch2):
#         def truncate(tensor):
#             # Apply batch size truncation
#             if self.final_batch_size is not None:
#                 tensor = tensor[:self.final_batch_size]
#             # Apply sequence length truncation (assuming sequences are at dim 1)
#             if self.final_seq_len is not None and tensor.ndim > 1:
#                 tensor = tensor[:, :self.final_seq_len]
#             return tensor

#         if isinstance(batch1, tuple):
#             combined = tuple(torch.cat([b1, b2]) for b1, b2 in zip(batch1, batch2))
#             return tuple(truncate(t) for t in combined)
#         elif isinstance(batch1, dict):
#             combined = {k: torch.cat([batch1[k], batch2[k]]) for k in batch1}
#             return {k: truncate(v) for k, v in combined.items()}
#         else:
#             combined = torch.cat([batch1, batch2])
#             return truncate(combined)

In [28]:
import numpy as np
def trunc_collate(batch, seq_len=None, batch_size=None):
    truncated = [sequence[:seq_len] for sequence in batch[:batch_size]]
    return torch.from_numpy(np.array(truncated))

In [10]:
comb = CustomCombinedLoader(dataloader_1, dataloader_2, 20, 12 )

In [11]:
oks = next(iter(comb))
oks

tensor([[  312, 37217,   284,  7301,   262,  1440, 14371,   286,   262,  5440,
            13,   887,   287,   281,  2479,   810,  3067,   318,   517,  9857],
        [ 6593,  5952,   287, 14493,    11,   968,  1971,    11,   290, 10140,
           389,  1022,   642,    25,    16,   290,   767,    25,    16,    13],
        [ 1002,   523,    11,   345,   389,   287,   262,   826,  1295,    11,
           356,  2630,   428,  1281,   329,   345,     0,   554,  2142,   352],
        [ 8472,  8557,  2891,    13, 25134,  9084,  1077,   291, 24075,    11,
          8798,   262, 10074,   286,  3406,  8653,    11,   318,   763,    12],
        [  632,  7176,   484,   481,  2298,   510,   511,  6569,  6973,    11,
          1803,   807,   319,  3232,  1177,    11, 19048,   319,  5274,   393],
        [19575,  6079,  6071,   517,  2829, 10345,   871,   290,  2836,    12,
         13120,  7071,    13,   775,  1239,  2245,  4856,   674,  2891,   290],
        [  286, 14457,    26,   290,   607,  3

In [19]:
oks.shape

torch.Size([12, 20])

In [23]:
next(iter(dataloader_1))

tensor([[  312, 37217,   284,  7301,   262,  1440, 14371,   286,   262,  5440,
            13,   887,   287,   281,  2479,   810,  3067,   318,   517,  9857,
           621,  1683,    11,   644,   857,   340,  1107,  1612,   284,   423,
           587,   564,   246],
        [ 6593,  5952,   287, 14493,    11,   968,  1971,    11,   290, 10140,
           389,  1022,   642,    25,    16,   290,   767,    25,    16,    13,
          2893, 41432, 26297,  6593,   743,  6133,  2585,   287,  8868, 13594,
         41528,   592,   290],
        [ 1002,   523,    11,   345,   389,   287,   262,   826,  1295,    11,
           356,  2630,   428,  1281,   329,   345,     0,   554,  2142,   352,
            11,   356,   447,   247,   260,  1016,   284,  2962,   319,   617,
           286,   262,  1243],
        [ 8472,  8557,  2891,    13, 25134,  9084,  1077,   291, 24075,    11,
          8798,   262, 10074,   286,  3406,  8653,    11,   318,   763,    12,
         39351,   351,   607,  6621,  

In [13]:
next(iter(dataloader_2))

tensor([[  290, 38082,  2569,  6650,   326,  1450,   815,   307,   287,  3877,
           286,   262,  2099,   286,  1535,  1337,  2594,   257,  2415,   423,
          1895,   284,    13,   198,  9444,  4234, 21916,   507,   351, 42153,
           284, 31572,   425],
        [   11,   262,  4095,  3496,   318,  4753,   319,  2046,    13,   887,
           644,  1838,   428,  3496,  5552,  3241,  8688,   318,   326,   673,
          3181,   656,  1657,   644,   347, 31777,  7259,   561,   804,   588,
           611,   484,   547],
        [  355,   340, 12850,  2854, 22104, 15193,  2449,   631,  9670,    13,
          1881,   989,   287,  2807,  6714,  5644,   326, 10691,   329,  3999,
          6403,  1466,   318,   366, 26069,  2249,     1,   290,   366,    83,
          1124,   670,     1],
        [  481,  1309,   345,  2251,    11,  1998,    11,   290,  2648, 31403,
           513,    35,  2008,   286,  1103,   661,   851,   329,  7166,  3950,
            11, 30259,  3950,    11,  

In [32]:

trunc_dataloader = get_dataloader(
    dataset_type="c4",
    dataset_path="data",
    dataset_split="train",
    total_batch_size=13,
    sequence_length=32,
    num_workers=0,
    seed=1,
    world_size_independent=False,
    use_new_sampling_method=True,
    shuffle=True,
    collate_fn=lambda x: trunc_collate(x, seq_len=20, batch_size=12),
)

In [33]:
next(iter(trunc_dataloader)).shape

torch.Size([12, 20])

In [41]:
from datasets import load_from_disk
path = "data"
dataset = load_from_disk(path).to_iterable_dataset(num_shards=64)

In [47]:
next(iter(dataset))

{'text': 'Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.\nHe will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.\nThe cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.',
 'timestamp': '2019-04-25 12:57:54',
 'url': 'https://klyq.com/beginners-bbq-class-taking-place-in-missoula/'}

In [48]:
dataset = dataset.shuffle(buffer_size=100, seed=1)

In [51]:
next(iter(dataset))

{'text': 'The ebook makes a speciality of geological historical past because the severe think about making a choice on the current biodiversity and landscapes of Amazonia. the various riding mechanisms for panorama evolution are explored through reviewing the historical past of the Amazonian Craton, the linked sedimentary basins, and the position of mountain uplift and weather swap.\nThis e-book provdes an perception into the Meso- and Cenozoic list of Amazonia that used to be characterised by way of fluvial and long-lived lake platforms and a hugely varied wildlife. This fauna comprises giants akin to the ca. 12 m lengthy caiman Purussaurus, but additionally a diversified fish fauna and fragile molluscs, while fossil pollen and spores shape relics of ancestral swamps and rainforests.\nWith exceptional wit, readability, and intelligence, Richard Dawkins, one of many world&apos;s most famed evolutionary biologists, has brought numerous readers to the wonders of technological know-how in

In [53]:
from transformers import GPT2TokenizerFast, PreTrainedTokenizerBase
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
def _tokenize_document(example):
    """Tokenize document and add end-of-text token."""
    tokens = tokenizer.encode(example["text"]) +  tokenizer.encode("<|endoftext|>")
    return {"tokens": tokens}

In [55]:
dt = dataset.map(_tokenize_document)

In [59]:
next(iter(dt))

{'text': 'The ebook makes a speciality of geological historical past because the severe think about making a choice on the current biodiversity and landscapes of Amazonia. the various riding mechanisms for panorama evolution are explored through reviewing the historical past of the Amazonian Craton, the linked sedimentary basins, and the position of mountain uplift and weather swap.\nThis e-book provdes an perception into the Meso- and Cenozoic list of Amazonia that used to be characterised by way of fluvial and long-lived lake platforms and a hugely varied wildlife. This fauna comprises giants akin to the ca. 12 m lengthy caiman Purussaurus, but additionally a diversified fish fauna and fragile molluscs, while fossil pollen and spores shape relics of ancestral swamps and rainforests.\nWith exceptional wit, readability, and intelligence, Richard Dawkins, one of many world&apos;s most famed evolutionary biologists, has brought numerous readers to the wonders of technological know-how in

In [61]:
from torch.utils.data import IterableDataset, DataLoader
dataloader = DataLoader(
    dt,
    batch_size=10,
    # collate_fn=collate_fn,
    # pin_memory=True,
    num_workers=0,
)

In [62]:
next(iter(dataloader))

RuntimeError: each element in list of batch should be of equal size

In [22]:
dataset = C4Dataset(
    sequence_length=13 + 1,
    split="train",
    path="data",
    seed=12,
    use_new_sampling_method=True,
    shuffle=False,
    world_size_independent=False,
)

In [ ]:
next(iter(dataset))